[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_C.ipynb)


# Set C — Passive Membrane (RC & Cable)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair
## Table of Contents
- [C1 Nernst → e_pas](#c1)
- [C2 Step response & τ](#c2)
- [C3 Impulse‑like kernel](#c3)
- [C4 Sinusoidal frequency response](#c4)
- [C5 R_in vs g_pas](#c5)
- [C6 Dendritic attenuation](#c6)
- [C7 Estimate λ](#c7)
- [C8 Sealed‑end vs length](#c8)
- [C9 Temporal & spatial summation](#c9)
- [C10 Parameter sweep](#c10)

This notebook extends Sets A and B by focusing **only** on the passive membrane: a leak conductance and a membrane capacitance.

### **C0 - Starter / Initialization**
**Purpose:** 
*   Create a **passive single-compartment soma** model (including leak conductance and membrane capacitance).
*   Initialize **recorders** for voltage and time.
*   Define a standardized `run()` utility to ensure consistent plotting across all simulations.

---

### **Post-Run Checklist**
After executing the initialization cell, please record the following in your notes:
1.  **NEURON Version:** Note the version number printed during the import process.
2.  **Import Messages:** Document any warnings or status messages (e.g., "NEURON successfully imported").

---

### **Exercises**

#### **1. Clamp Definitions**
In one concise sentence each, define the following experimental setups:
*   **Current Clamp:** You command a specific current ($I$) and observe the resulting changes in membrane potential ($V_m$).
*   **Voltage Clamp:** You command a specific membrane potential ($V$) and observe the resulting current ($I$) required to maintain it.

#### **2. Notebook Setup & Defaults**
*   **Identification:** Add your **initials** and the **current date** to a new header cell at the very top of the notebook.
*   **Global Defaults:** List all global parameters you are relying on for this model. Include values for:
    *   **$g_{\text{pas}}$**: [Enter Value]
    *   **$e_{\text{pas}}$**: [Enter Value]
    *   **$R_a$**: [Enter Value]
    *   **$c_m$**: [Enter Value]

#### **3. Reproducibility Requirement**
**Question:** Why is plotting units (e.g., ms, mV, nA) on axes considered a fundamental requirement for **reproducibility**?  
*   [Your Answer Here]

In [ ]:
# --- Set C Starter: Install NEURON and create a passive single-compartment cell ---
!pip -q install neuron==8.2.4
from neuron import h, gui
from neuron.units import ms, mV
import matplotlib.pyplot as plt
import numpy as np

h.load_file("stdrun.hoc")

# Create a spherical soma (single compartment)
soma = h.Section(name='soma')
soma.L = 20    # µm
soma.diam = 20 # µm
soma.Ra = 100  # ohm·cm

# Insert passive leak only
soma.insert('pas')
soma.g_pas = 0.0002     # S/cm^2 (leak conductance density)
soma.e_pas = -65        # mV (leak reversal)
soma.cm    = 1.0        # uF/cm^2

# Recorders
t = h.Vector().record(h._ref_t)
v = h.Vector().record(soma(0.5)._ref_v)

# Utility to run and plot
def run(sim_ms=60, v_init=-65, title="", figsize=(6,3)):
    h.finitialize(v_init*mV)
    h.continuerun(sim_ms*ms)
    plt.figure(figsize=figsize)
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True, alpha=0.25); plt.show()

print('Set C starter ready — passive membrane initialized.')



### **C1 - Nernst $\to$ Leak Reversal (Passive $E_L$)**
**Idea:** 
Physiological concentrations define ion-specific **Nernst potentials**. A passive leak’s reversal potential can be set near an ion’s equilibrium to illustrate the **resting membrane level** and **driving force** without the complexity of active channels.

---

### **What to Change**
In the Nernst helper function, you will be able to:
*   Adjust **inside/outside concentrations** ($[C]_{in}$ and $[C]_{out}$).
*   Modify the **temperature** ($T$).
*   Update the model's `e_pas` to match your calculated Nernst potential.

---

### **Sanity Checks**
1.  **Print Calculated Potentials:** Confirm the values for $E_{\text{K}}$, $E_{\text{Na}}$, and $E_{\text{Cl}}$.
2.  **Resting Level Check:** Run a small negative pulse to confirm the resting level settles near the chosen `e_pas`.

---

### **Predict $\to$ Verify**
*   **Driving Force Logic:** If the holding potential is more depolarized than $E_{\text{Cl}}$, a $\text{Cl}^-$-like conductance should appear **outward** (and vice-versa), driven purely by the electrochemical gradient.

---

### **Exercises**

#### **1. Temperature Dependence**
Compute the Nernst potential for $\text{K}^+$ at **27 °C** and **37 °C**. Explain the numeric difference between these two values.

#### **2. Steady-State Injection**
Set `e_pas` to your computed $E_{\text{K}}$. Inject a tiny positive current step ($I$) and report the resulting steady-state voltage level compared to $E_{\text{K}}$.

#### **3. Conceptual Understanding**
Explain, in two sentences, why **reversal potential** is a property of the specific conductance (ion channel type) rather than a property of the cell membrane itself.


In [ ]:
# Helper: Nernst potential (base-10) at temperature T_C (Celsius)
R = 8.31446261815324  # J/(mol·K)
F = 96485.33212       # C/mol

def nernst_mV(cin_mM, cout_mM, z=1, T_C=37.0):
    T_K = 273.15 + T_C
    # convert to base-10 form: 2.303*R*T/F ~ 61.5 mV at 37 C for z=1
    return (1000.0*R*T_K/(z*F))*np.log(cout_mM/cin_mM)  # natural log mV

for ion, params in {
    'K+':  {'cin':140, 'cout':5,  'z':1},
    'Na+': {'cin':12,  'cout':145,'z':1},
    'Cl-': {'cin':5,   'cout':120,'z':-1}
}.items():
    E = nernst_mV(params['cin'], params['cout'], params['z'], 37.0)
    print(f"E_{ion} ≈ {E:.1f} mV at 37°C")

# Demonstration: set leak reversal near potassium-like value
soma.e_pas = float(nernst_mV(140,5,1,37.0))
print('Set e_pas to ~E_K for demonstration:', soma.e_pas)

# Small negative pulse to see resting level near E_K
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=50; stim.amp=-0.03
run(sim_ms=70, title='C1: Passive cell with e_pas near E_K (Nernst review)')

# Restore classroom default for next steps
soma.e_pas = -65; stim.amp = 0.0

In [ ]:

!pip -q install neuron==8.2.4 >/dev/null 2>&1
try:
    from neuron import h
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np, matplotlib.pyplot as plt
print('Starter ready (NEURON expected).')


### **C2 - Step Response & Time Constant $\tau$**

**Idea:**  
A passive step response is exponential. The time constant, **$\tau$**, is defined as the time required for the membrane potential to reach **~63%** of the total distance between its initial state and its final steady state.

---

### **What to Change**
Experiment with the following parameters to observe their effect on the response:
*   **Stimulus:** Step amplitude ($I$) and duration.
*   **Membrane Properties:** Leak conductance ($g_{\text{pas}}$) and specific capacitance ($c_m$).

---

### **Sanity Checks**
1.  **Trace Measurement:** Manually compute $\tau$ from your recorded voltage trace.
2.  **Analytic Comparison:** Compare your measured $\tau$ to the theoretical estimate:  
    $$\tau \approx R_{\text{in}} \cdot C_m$$  
    *(Ensure $R_{\text{in}}$ is calculated correctly for your specific geometry.)*

---

### **Predict $\to$ Verify**
*   **Capacitance:** Increasing $c_m$ should **increase** $\tau$ (slower response).
*   **Conductance:** Increasing $g_{\text{pas}}$ should **decrease** $\tau$ (faster response).

---

### **Exercises**

#### **1. Capacitance Scaling**
Double the membrane capacitance ($c_m$) and re-measure $\tau$. Report the ratio between the new $\tau$ and the original.

#### **2. Conductance Manipulation**
Halve the leak conductance ($g_{\text{pas}}$) and re-measure both:
*   The new time constant ($\tau$).
*   The new steady-state change in voltage ($\Delta V$).
**Explain:** Why did these values change in the directions you observed?

#### **3. Visual Communication**
Suggest a **two-panel figure layout** that would effectively communicate the concept of $\tau$ to a reader. Describe what each panel would display.


In [ ]:

soma=h.Section(name='soma'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=60; stim.amp=-0.05
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(80*ms)
import numpy as np
T=np.array(t); V=np.array(v)
ss=(T>60)&(T<63); vss=V[ss].mean(); v0=V[np.where(T>=5)[0][0]]
vtgt=v0+0.632*(vss-v0)
idx=np.where(V<=vtgt)[0]
print('C2 τ ≈', (T[idx[0]]-5) if len(idx)>0 else float('nan'),'ms')
plt.figure(figsize=(7,3.3)); plt.plot(T,V,'k'); plt.title('C2: Step response')
plt.tight_layout(); plt.show()


### **C3 - Brief Pulse → Impulse-like Kernel of a passive memebrane**

**Idea:** A very brief current pulse approximates the **impulse response** (also known as the temporal kernel) of the passive membrane. This response represents how the membrane "filters" fast, discrete inputs.

---

### **What to Change**
Modify the following to observe changes in the membrane response:
*   **Pulse Dynamics:** Adjust the pulse **width** and **amplitude**.
*   **Initial State:** Change the **baseline level** of the membrane potential.

---

### **Checks**
1.  **Linearity Check:** Confirm that for small pulses, the **peak voltage** scales proportionally with the **pulse amplitude**. This verifies you are operating in the membrane's linear integration region.

---

### **Exercises**

#### **1. Charge Conservation**
Halve the pulse width while keeping the total **injected charge** constant (i.e., double the amplitude). 
*   Report any changes in the **peak voltage** and the **width** of the resulting response.

#### **2. Synaptic Integration**
Explain how this impulse-like kernel relates to **EPSP shaping**. Specifically, how would the membrane response differ for synapses with fast vs. slow time courses?
*   [Your Answer Here]



In [ ]:
stim.amp = -0.2; stim.dur = 1.0; stim.delay = 5
run(sim_ms=30, title='C3: Brief pulse response (impulse-like)')
# Restore
stim.amp=0.0

### **C4 - Sinusoidal frequency response**
Inject a sine current at different frequencies and observe attenuation/phase lag indicative of a first‑order low‑pass.
**Idea** The passive membrane acts as a first-order low-pass filter: higher-frequency inputs are naturally attenuated and phase-shifted.

---

### **What to change**
Sine frequency list: Adjust the range of frequencies to test.
Drive amplitude: Change the strength of the input signal.
Analysis window: Ensure the time window is long enough to capture several full cycles at lower frequencies.

---

### **Sanity checks**
Steady-state metrics: Report the steady-state amplitude and visible phase lag at 2, 10, and 50 Hz.
Monotonicity: Confirm that attenuation increases consistently as the frequency increases.

---

### **Exercises**

**Frequency Domain Plotting**
Plot the response amplitude versus frequency using a log-x axis across the range of 2–200 Hz.
Mark the −3 dB point on your plot.
Use this point to estimate the membrane time constant, $\tau$.

**Synaptic Selectivity**
Explain, in one paragraph, how this frequency selectivity interacts with typical synaptic kinetics. For example, how does the membrane's low-pass nature affect the integration of fast versus slow synaptic inputs?


In [ ]:
def sine_run(freq_hz=10, amp_nA=0.05, dur_ms=400):
    # Drive with a sine by updating an SEClamp or use a Vector.play to IClamp. Here: Vector.play to IClamp.
    stim.amp = 0.0
    ic = h.IClamp(soma(0.5)); ic.delay=0; ic.dur=1e9; ic.amp=0.0
    # time vector in ms
    tt = np.arange(0, dur_ms, 0.025)
    ii = amp_nA*np.sin(2*np.pi*freq_hz*(tt/1000.0))
    ivec = h.Vector(ii)
    tvec = h.Vector(tt)
    ivec.play(ic._ref_amp, tvec, 1)
    h.finitialize(-65*mV); h.continuerun(dur_ms*ms)
    # compute amplitude (peak-to-peak/2) after a transient
    t_np = np.array(t); v_np = np.array(v)
    mask = t_np > (dur_ms*0.5)
    v_amp = (v_np[mask].max() - v_np[mask].min())/2
    plt.figure(figsize=(6,3))
    plt.plot(t_np, v_np, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(f'C4: Sine drive {freq_hz} Hz, Vm amp ≈ {v_amp:.2f} mV')
    plt.grid(True, alpha=0.25); plt.show()

for f in [2, 10, 50]:
    sine_run(freq_hz=f, amp_nA=0.05, dur_ms=400)

### **C5 - $R_{\text{in}}$ vs. $g_{\text{pas}}$**   Vary `g_pas` and extract steady ΔV/ΔI to see how R_in changes (R_in ≈ ΔV/ΔI).

**Idea**
Input resistance ($R_{\text{in}}$) is calculated as $\Delta V / \Delta I$ at steady state. Increasing the leak conductance ($g_{\text{pas}}$) naturally lowers $R_{\text{in}}$, as more channels provide more pathways for current to escape the membrane.

---

### **What to change**
*   **Leak Conductance:** Test different values of $g_{\text{pas}}$.
*   **Stimulus:** Adjust the test pulse amplitude ($\Delta I$).

---

### **Sanity checks**
*   **Inverse Relation:** Produce a small table or list showing $g_{\text{pas}}$ vs. calculated $R_{\text{in}}$.
*   **Verification:** Confirm that as $g_{\text{pas}}$ increases, $R_{\text{in}}$ decreases.

---

### **Exercises**

**Numerical Validation of $\tau$**
For two different $g_{\text{pas}}$ values, estimate the time constant ($\tau$) directly from the step responses. 
*   Check if the relationship $\tau \approx R_{\text{in}} \cdot C_m$ holds numerically for your results.

**EPSP Scaling**
Explain, in one or two sentences, why a neuron with a higher $R_{\text{in}}$ exhibits larger EPSPs when subjected to identical synaptic conductances compared to a neuron with lower $R_{\text{in}}$.
*   [Your Answer Here]


In [ ]:
def measure_rin(gpas=0.0002, amp=-0.03):
    soma.g_pas = gpas
    stim.delay=5; stim.dur=40; stim.amp=amp
    h.finitialize(-65*mV); h.continuerun(60*ms)
    t_np = np.array(t); v_np = np.array(v)
    ss = (t_np>40) & (t_np<45)
    dv = v_np[ss].mean() - (-65)
    Rin = abs(dv/amp)  # mV/nA = MΩ
    return Rin, dv

vals = []
for g in [0.0001, 0.0002, 0.0005, 0.001]:
    Rin, dv = measure_rin(g)
    vals.append((g, Rin))
    print(f"g_pas={g:.5f} S/cm²  =>  R_in≈{Rin:.1f} MΩ (ΔV≈{dv:.2f} mV)")

soma.g_pas = 0.0002; stim.amp=0.0

### ** C6 - Dendritic attenuation & the space constant $\lambda$**  

Attach a passive dendrite, inject at the distal tip, and measure Vm at multiple positions to visualize attenuation.

**Idea**: A passive dendrite behaves as a cable; as a signal propagates, the membrane potential ($V_m$) attenuates with distance from the source. This decay is characterized by the space constant ($\lambda$).

---

### **What to change**
*   **Morphology:** Adjust the dendrite length and diameter.
*   **Experimental Setup:** Change the current injection site (e.g., soma, mid-shaft, or distal tip).
*   **Cable Properties:** Modify the leak conductance ($g_{\text{pas}}$) and axial resistivity ($R_a$).

---

### **Sanity checks**
*   **Spatial Mapping:** Plot voltage traces at multiple positions along the dendrite simultaneously.
*   **Propagation Effects:** Confirm that distal responses appear both smaller in amplitude and broader in time (filtered) when recorded at the soma compared to the local injection site.

---

### **Exercises**

**Morphology and Attenuation**
Increase the dendrite diameter and observe the changes in the somatic EPSP amplitude and width. 
*   **Explain:** Use your intuition of cable theory to justify why a thicker dendrite changes the signal this way.

**Injection Site Influence**
Move the current injection point from the distal tip to the mid-shaft of the dendrite. 
*   **Describe:** How does this change the amplitude and the "rise time" of the signal recorded at the soma?


In [ ]:
# Build dendrite and record at several locations
try:
    dend = dend
except NameError:
    dend = h.Section(name='dend')
    dend.L = 800; dend.diam = 1.5; dend.Ra = 100
    dend.insert('pas'); dend.g_pas = soma.g_pas; dend.e_pas = soma.e_pas
    dend.connect(soma(1.0))

sites = [0.0, 0.25, 0.5, 0.75, 1.0]
recs  = [h.Vector().record(dend(x)._ref_v) for x in sites]

stim_d = h.IClamp(dend(1.0)); stim_d.delay=5; stim_d.dur=40; stim_d.amp=-0.05
h.finitialize(-65*mV); h.continuerun(70*ms)

plt.figure(figsize=(6,3))
for x,vec in zip(sites,recs):
    plt.plot(t, vec, label=f'dend({x:.2f})')
plt.plot(t, v, label='soma(0.5)', lw=2, color='k')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C6: Passive attenuation along dendrite')
plt.legend(); plt.grid(True, alpha=0.25); plt.show()

## C7 — Estimate the Space Constant λ from the Steady Profile

Fit an exponential to the steady‑state voltage vs distance along the dendrite.
### Estimate λ from steady-state profile

**Idea**
At steady state, the voltage decay along a passive cable $V(x) - V_{\text{rest}}$ is approximately exponential with distance. By fitting $\ln|V(x) - V_{\text{rest}}|$ vs. $x$, we can estimate the space constant $\lambda$ from the resulting slope.

---

### What to change
Sampling locations: Choose different points along the dendrite to record voltage.
Pulse duration/size: Ensure the pulse is long enough to reach a true steady state.
Morphology and Biophysics: Experiment with different axial resistivity ($R_a$) and diameter values.

---

### Sanity checks
Report the fitted $\lambda$: Calculate the value for your current setup.
Geometric comparison: Compare the $\lambda$ values obtained across two different dendritic geometries.

---

### Exercises

**Diameter Scaling**
Halve the dendrite diameter and re-estimate $\lambda$. 
*   **Explain:** Describe the direction of change and why this occurs based on cable theory.

**Branching Assumptions**
Discuss one specific assumption used in this simplified linear fit that would break down if you were modeling a complex branching tree instead of a single cable.
*   [Your Answer Here]

Discuss one assumption in this didactic fit that would break down in a branching tree.

In [ ]:
# Sample steady levels at locations and fit V(x)=V0*exp(-x/lambda)
locs_um = np.array([0, 200, 400, 600, 800])
# Record new steady levels using a long pulse
stim_d.delay=5; stim_d.dur=200; stim_d.amp=-0.05
h.finitialize(-65*mV); h.continuerun(220*ms)

vals = []
for x in [0.0, 0.25, 0.5, 0.75, 1.0]:
    vec = h.Vector(); vec.record(dend(x)._ref_v)
# Re-run to fill new vectors (simple approach)
h.finitialize(-65*mV); h.continuerun(220*ms)

# Actually compute steady values using existing recs
steady = []
for x in [0.0, 0.25, 0.5, 0.75, 1.0]:
    vec = h.Vector(); vec.record(dend(x)._ref_v)
# We need a clean re-sim to fill; do once more
h.finitialize(-65*mV); h.continuerun(220*ms)

# For simplicity, just sample from previous recs list
steady = []
for vec in recs:
    arr = np.array(vec)
    steady.append(arr[-50:].mean())

V_soma = np.array(v)[-50:].mean()
V_dist = np.array(steady)
xs = np.array([0, 0.25, 0.5, 0.75, 1.0]) * 800.0  # convert to µm

# Fit log-linear: ln(|V(x)-V_rest|) ~ -x/lambda + c
Vrest = -65.0
mag = np.abs(V_dist - Vrest) + 1e-6
coef = np.polyfit(xs, np.log(mag), 1)
lambda_um = -1.0/coef[0]
print(f"Estimated λ ≈ {lambda_um:.1f} µm (didactic fit)")

plt.figure(figsize=(5,3.5))
plt.plot(xs, mag, 'o', label='data')
plt.plot(xs, np.exp(coef[1] + coef[0]*xs), '-', label='exp fit')
plt.xlabel('Distance (µm)'); plt.ylabel('|V - V_rest| (mV)'); plt.title('C7: λ estimate from steady attenuation')
plt.legend(); plt.grid(True, alpha=0.25); plt.show()

## C8 — Sealed‑End Effects vs Longer Cable

Compare a shorter vs longer dendrite (sealed end) to see how boundary conditions affect attenuation.
### **C8 - Sealed-end vs. longer cable**

**Idea**
Boundary conditions matter: a sealed tip and longer lengths modify attenuation and waveform shape. Signals reflect at high-resistance boundaries, such as a sealed terminal, which can bolster local voltage at the cost of propagating less efficiently.

---

### **What to change**
Length values: Experiment with the total length of the cable.
Keep other parameters fixed: Ensure $g_{\text{pas}}$, $c_m$, and $R_a$ remain constant to isolate length effects.

---

### **Sanity checks**
Overlay soma vs tip traces: Compare the recordings for two different cable lengths.
Difference analysis: Comment on how the attenuation and temporal filtering (slowing of the rise time) differ between the short and long cables.

---

### **Exercises**

**Qualitative Explanations**
Provide a qualitative explanation of how reflections at sealed ends might affect transients compared to an "infinite" cable where no reflection occurs.
*   [Your Answer Here]

**Experimental Design**
Propose a minimal experiment to distinguish "true length" effects (morphology) from changes in internal parameters like axial resistivity ($R_a$).
*   [Your Answer Here]


In [ ]:
def run_dend(L_um=400):
    d = h.Section(name=f'dend_{L_um}')
    d.L = L_um; d.diam = 1.5; d.Ra = 100
    d.insert('pas'); d.g_pas=soma.g_pas; d.e_pas=soma.e_pas
    d.connect(soma(1.0))
    vtip = h.Vector().record(d(1.0)._ref_v)
    stimx = h.IClamp(d(1.0)); stimx.delay=5; stimx.dur=40; stimx.amp=-0.05
    h.finitialize(-65*mV); h.continuerun(70*ms)
    return np.array(t), np.array(v), np.array(vtip)

for L in [300, 800, 1500]:
    tt, vsoma, vtip = run_dend(L)
    plt.figure(figsize=(6,3))
    plt.plot(tt, vsoma, label='soma')
    plt.plot(tt, vtip, label=f'tip (L={L} µm)')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title('C8: Sealed-end length effects')
    plt.legend(); plt.grid(True, alpha=0.25); plt.show()

## C9 — Temporal and Spatial Summation in a Passive Cell

### **C9 - Temporal & spatial summation**
Deliver a train of small inputs (temporal) and two simultaneous inputs at different sites (spatial).

**Idea**: Passive summation depends on timing and location: inputs occurring close in time sum more strongly due to the membrane's time constant. Similarly, proximal inputs contribute more to the somatic potential than distal inputs of the same size because they undergo less dendritic attenuation.

---

### **What to change**
Pulse train interval: Adjust the time between successive inputs to test temporal integration.
Two injection sites: Experiment with simultaneous or staggered inputs at different locations (e.g., soma and dendrite).

---

### **Sanity checks**
Summation Additivity: Identify the shortest inter-pulse interval that yields approximately additive summation.
Spatial Comparison: Compare the somatic response of simultaneous distal injections at different sites versus a single proximal injection.

---

### **Exercises**

**Effect of τ on Summation**
Increase the membrane time constant ($\tau$) and observe how it alters the degree of temporal summation at the soma. 
*   **Explain:** Why does a longer $\tau$ lead to a more "integrated" signal?

**Sublinear Spatial Summation**
Demonstrate that two identical distal inputs can sum sublinearly at the soma (where the total is less than the sum of their individual parts) compared with proximal inputs. 
*   **Explain:** Discuss why the location of the inputs relative to each other and the soma causes this sublinear effect.


In [ ]:
# Temporal summation at soma
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=3; stim.amp=0.05
# play a train via Vector.play
tt = np.arange(0, 120, 0.1)
ii = np.zeros_like(tt)
for k in range(10):
    onset = 10 + k*5 # ms
    ii[(tt>=onset) & (tt<onset+1)] = 0.05
ivec = h.Vector(ii); tvec = h.Vector(tt)
ivec.play(stim._ref_amp, tvec, 1)
h.finitialize(-65*mV); h.continuerun(120*ms)
plt.figure(figsize=(6,3))
plt.plot(t, v, 'k'); plt.title('C9: Temporal summation (soma)')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.grid(True, alpha=0.25); plt.show()

# Spatial summation: two distal sites on dendrite
ic1 = h.IClamp(dend(0.8)); ic1.delay=10; ic1.dur=5; ic1.amp=-0.05
ic2 = h.IClamp(dend(0.2)); ic2.delay=10; ic2.dur=5; ic2.amp=-0.05
h.finitialize(-65*mV); h.continuerun(40*ms)
plt.figure(figsize=(6,3))
plt.plot(t, v, 'k'); plt.title('D9: Spatial summation (dend(0.2)+dend(0.8))')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.grid(True, alpha=0.25); plt.show()

## C10 — Playground: Parameter Sweep for τ, R_in, and Attenuation


### **C10 - Parameter sweep ($\tau$, $R_{\text{in}}$, attenuation)**
Sweep membrane parameters (cm, g_pas, dend diameter) and observe effects on τ, R_in, and attenuation.

**Idea**: Systematically vary membrane properties—specifically capacitance ($c_m$), leak conductance ($g_{\text{pas}}$), and dendrite diameter—to record the resulting time constant ($\tau$), input resistance ($R_{\text{in}}$), and a spatial attenuation metric. This allows for a comprehensive mapping of how biophysical parameters intuitively shape signal integration.

---

### **What to change**
*   **Three-tuple settings:** Systematically adjust $c_m$, $g_{\text{pas}}$, and diameter according to the provided combinations.
*   **Custom configuration:** Add one parameter set of your own design to test a specific hypothesis (e.g., a "leaky" vs. "tight" membrane).

---

### **Sanity checks**
Confirm that your results follow the expected biological trends:
*   **Capacitance:** $\uparrow c_m \rightarrow \uparrow \tau$ (Slower membrane charging).
*   **Leak:** $\uparrow g_{\text{pas}} \rightarrow \downarrow R_{\text{in}}$ (Lower input resistance).
*   **Diameter:** $\uparrow \text{diameter} \rightarrow \downarrow \text{attenuation}$ (Better signal propagation).

---

### **Exercises**

**Attenuation Indexing**
Update your simulation output to include a calculated **Attenuation Index** (e.g., the steady-state ratio of the voltage at the distal tip compared to the voltage at the soma). 
*   Report this index for each of your parameter configurations.

**Coincidence Detection Constraints**
In two sentences, explain how the time constant ($\tau$) and the space constant ($\lambda$) jointly constrain the "window" for **coincidence detection**. 
*   Compare how these constraints differ for a pair of distal synapses versus a pair of proximal synapses.


In [ ]:
def passive_summary(cm=1.0, gpas=0.0002, diam=1.5):
    soma.cm = cm; soma.g_pas = gpas
    dend.diam = diam
    # τ from D2-like pulse
    stim.delay=5; stim.dur=40; stim.amp=-0.05
    h.finitialize(-65*mV); h.continuerun(60*ms)
    t_np = np.array(t); v_np = np.array(v)
    ss = (t_np>40)&(t_np<45)
    v_ss = v_np[ss].mean()
    v0 = v_np[np.where(t_np>=5)[0][0]]
    vtgt = v0 + 0.632*(v_ss - v0)
    idx = np.where(v_np <= vtgt)[0]
    tau_est = (t_np[idx[0]] - 5.0) if len(idx)>0 else np.nan
    Rin = abs((v_ss - v0)/(-0.05))
    return tau_est, Rin

settings = [
    (1.0, 0.0002, 1.5),
    (2.0, 0.0002, 1.5),
    (1.0, 0.0005, 1.5),
    (1.0, 0.0002, 3.0),
]
for cm,gpas,diam in settings:
    tau_est, Rin = passive_summary(cm, gpas, diam)
    print(f"cm={cm}, g_pas={gpas}, dend.diam={diam}  =>  tau≈{tau_est:.2f} ms, R_in≈{Rin:.1f} MΩ")

# Restore defaults
soma.cm=1.0; soma.g_pas=0.0002; dend.diam=1.5; stim.amp=0.0

---

## Practice / Discussion Questions — Set C — Passive Membrane & Cable

1) Define τ and λ in words, and give one empirical method to estimate each.
2) *Predict*: If dendritic diameter decreases, what happens to λ and distal EPSP amplitude at the soma?
3) Why does an EPSP recorded locally look different from the somatic EPSP for the same input event?
4) Show qualitatively how **branch points** can affect attenuation.
5) Explain why the passive membrane behaves like a **low‑pass filter**; connect to synaptic frequency content.
6) *Reasoning*: How does increasing membrane resistance affect τ and EPSP area?
7) Give one case where **apparent linear summation** fails and what it suggests mechanistically.
8) Provide one realistic parameter set (R_m, C_m, R_a) and describe expected EPSP shape qualitatively.
9) Why might a **distal EPSP** be broader at the soma than a proximal EPSP?
10) *Compute*: If τ doubles, what happens to EPSP half‑width at the soma (qualitative)?
11) Explain **electrotonic distance** vs physical distance.
12) Identify a **diagnostic plot** (e.g., transfer impedance vs distance) that reveals cable effects.
13) Why can passive properties alone **mislead** about synaptic strength across locations?
14) *Design*: A single‑compartment model fits soma data but not dendritic data. What’s the next minimal step?
15) What are the **limits** of passive models, and what signals adding active conductances?
16) Explain how τ and λ together capture **temporal vs spatial** integration aspects.
17) Describe how **holding potential** can change apparent PSP size in different compartments.
18) Provide one **artifact** that could confound cable measurements and how to control it.
19) *Synthesis*: State the minimal passive features to keep for Set E timing experiments.
20) Summarize how Set C prepares you for timing motifs and location dependence later.
